# 7. Input classification: embeddings vs the LLM

The cardholder chat classifies each message into one of eight intents (`guardrails.classify`). Today that is rules, then an LLM call, then keywords. This notebook asks whether a **frozen sentence encoder plus a small classifier** (all-MiniLM-L6-v2 and logistic regression) could replace the LLM call for this job: no extra network round trip, no data sent out, and the same answer every time.

**Data.** `intent_dataset.json` holds 160 training sentences (20 per intent) and a separate **hard test set** of 80 sentences written in a different voice: indirect, lowercase, typos, longer. Nothing in the rules or the classifier was tuned on the hard set, so it measures generalisation. Cross-validation on the training set is reported too, but it is optimistic because the sentences share a style.

**What matters most is not accuracy.** A missed attack (`sensitive_data` or `prompt_injection` classed as something harmless) is far worse than a wrongly refused question. So each method is scored on accuracy, macro F1, **attack recall** and **false-refusal rate**.

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.ai import guardrails

SEED = 42
LABELS = list(guardrails.INTENTS)
ATTACKS = {guardrails.SENSITIVE_DATA, guardrails.PROMPT_INJECTION}

dataset = json.loads((PROJECT_ROOT / "notebooks" / "intent_dataset.json").read_text(encoding="utf-8"))
train = pd.DataFrame(dataset["train"])
hard = pd.DataFrame(dataset["hard_test"])
print(len(train), "training sentences,", len(hard), "hard-test sentences")
train["label"].value_counts().to_frame("per intent").T

## 1. Encode

all-MiniLM-L6-v2 is a 22M-parameter encoder that maps a sentence to a 384-number vector. It is used frozen: no fine-tuning, so the only thing learned from our data is the small classifier on top.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")


def embed(texts):
    return encoder.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)


train_vectors = embed(train["text"])
hard_vectors = embed(hard["text"])
train_vectors.shape

## 2. Choose the regularisation on the training set only

Five-fold cross-validation picks `C`. The hard set is not touched here.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = []
for strength in (0.1, 1, 10, 100):
    model = LogisticRegression(C=strength, max_iter=2000, random_state=SEED)
    predicted = cross_val_predict(model, train_vectors, train["label"], cv=folds)
    search.append({"C": strength, "cv_accuracy": accuracy_score(train["label"], predicted)})
search = pd.DataFrame(search)
BEST_C = float(search.sort_values("cv_accuracy", ascending=False).iloc[0]["C"])
print("chosen C =", BEST_C)
search.round(3)

In [ ]:
classifier = LogisticRegression(C=BEST_C, max_iter=2000, random_state=SEED).fit(train_vectors, train["label"])
probabilities = classifier.predict_proba(hard_vectors)
embedding_predicted = classifier.classes_[probabilities.argmax(axis=1)]
confidence = probabilities.max(axis=1)

## 3. Compare on the hard set

Four methods, all on the same 80 unseen sentences:

- **rules + keywords**: what runs today with no model (`rule_intent`, else `keyword_intent`).
- **embedding**: MiniLM + logistic regression alone.
- **rules + embedding**: the deterministic rules first (they are a floor that never depends on a model), the embedding classifier for everything else. This is the candidate replacement.
- **LLM**: optional, off by default (section 6).

In [ ]:
def summarise(name, predicted, truth=hard["label"]):
    truth = np.asarray(truth)
    predicted = np.asarray(predicted)
    is_attack = np.isin(truth, list(ATTACKS))
    refused = np.isin(predicted, list(ATTACKS))
    return {
        "method": name,
        "accuracy": accuracy_score(truth, predicted),
        "macro_f1": f1_score(truth, predicted, average="macro"),
        "attack_recall": refused[is_attack].mean(),
        "false_refusal_rate": refused[~is_attack].mean(),
    }


rules_keywords = [guardrails.rule_intent(text) or guardrails.keyword_intent(text) for text in hard["text"]]
rules_embedding = [
    guardrails.rule_intent(text) or predicted
    for text, predicted in zip(hard["text"], embedding_predicted)
]

comparison = pd.DataFrame([
    summarise("rules + keywords", rules_keywords),
    summarise("embedding", embedding_predicted),
    summarise("rules + embedding", rules_embedding),
]).set_index("method")
comparison.round(3)

Cross-validation on the training style, for reference only. The gap to the hard-set accuracy is how much the shared style flatters it.

In [ ]:
cv_predicted = cross_val_predict(
    LogisticRegression(C=BEST_C, max_iter=2000, random_state=SEED), train_vectors, train["label"], cv=folds
)
summarise("embedding, 5-fold CV on train", cv_predicted, truth=train["label"]).get("accuracy")

## 4. Where it goes wrong

In [ ]:
errors = hard.assign(embedding=embedding_predicted, confidence=confidence.round(2))
errors = errors[errors["label"] != errors["embedding"]]
pd.set_option("display.max_colwidth", 90)
errors[["text", "label", "embedding", "confidence"]]

In [ ]:
matrix = confusion_matrix(hard["label"], embedding_predicted, labels=LABELS)
figure, axis = plt.subplots(figsize=(6.5, 5.5))
axis.imshow(matrix, cmap="Blues")
axis.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
axis.set_yticks(range(len(LABELS)), LABELS)
for row in range(len(LABELS)):
    for column in range(len(LABELS)):
        if matrix[row, column]:
            axis.text(column, row, matrix[row, column], ha="center", va="center")
axis.set_xlabel("predicted")
axis.set_ylabel("true")
axis.set_title("Embedding classifier on the hard set")
plt.tight_layout()

## 5. A cheaper hybrid: abstain when unsure

Use the embedding classifier when it is confident, and send only the uncertain messages to the LLM. The sweep shows, for each confidence threshold, how many messages the embedding classifier would keep (coverage) and how accurate it is on those.

In [ ]:
sweep = []
for threshold in (0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    kept = confidence >= threshold
    sweep.append({
        "threshold": threshold,
        "coverage": kept.mean(),
        "accuracy_on_kept": accuracy_score(hard["label"][kept], embedding_predicted[kept]) if kept.any() else np.nan,
        "sent_to_llm": int((~kept).sum()),
    })
pd.DataFrame(sweep).round(3)

## 6. Optional: the LLM itself

Off by default: it sends the 80 hard-test sentences to whichever model is configured (OpenAI if `OPENAI_API_KEY` is set, which is billed, otherwise local Ollama). It calls the classifier prompt directly, skipping the rules, so it measures the model alone.

In [ ]:
RUN_LLM = False

if RUN_LLM:
    from app.ai import llm_client

    llm_predicted, llm_latencies = [], []
    for text in hard["text"]:
        call = llm_client.chat([
            {"role": "system", "content": guardrails.CLASSIFIER_PROMPT},
            {"role": "user", "content": text},
        ])
        llm_latencies.append(call.latency_ms)
        llm_predicted.append(guardrails._parse_label(call.text) or "unparsed")
    llm_row = summarise("LLM alone", llm_predicted)
    llm_row["unparsed"] = sum(1 for label in llm_predicted if label == "unparsed")
    print(llm_row)
    print("median latency ms:", int(np.median(llm_latencies)))

## 7. Speed

In [ ]:
single = hard["text"].iloc[0]
embed([single])  # warm-up
started = time.perf_counter()
for _ in range(50):
    classifier.predict(embed([single]))
per_message_ms = (time.perf_counter() - started) / 50 * 1000
print(f"embedding + classifier: {per_message_ms:.1f} ms per message on CPU")

## 8. Reading the result

*(Filled in after running; see the results above.)*